# 13 — Security: Validation, Safe Parsing, Secrets and Dependencies

Advanced Python needs secure boundaries: avoid arbitrary code execution, untrusted deserialization, command injection, leaked credentials, and accidental denial-of-service inputs.

*Advanced · 90–150 min*

## You will learn

- Choose json rather than eval for external data
- Explain pickle trust boundaries and subprocess safety
- Use secrets and constant-time comparisons appropriately
- Validate and bound untrusted input

## Examples

### 1. JSON is data, not code

json.loads parses JSON values without evaluating arbitrary Python expressions. It still needs schema validation, size limits, and domain checks. Never eval user-supplied JSON-like strings.

In [ ]:
import json

raw = '{"name": "Ada", "count": 3}'
data = json.loads(raw)
if not isinstance(data, dict) or not isinstance(data.get("count"), int):
    raise ValueError("unexpected data")
print(data["name"], data["count"])

**Design review:** Could a syntactically valid JSON file still cause a memory exhaustion problem?

### 2. Pickle is a trust boundary

pickle can execute code while loading malicious serialized objects. Use it only for trusted data from trusted sources, and don't load untrusted .pkl files or attachments. JSON is better for interoperable public data.

In [ ]:
import pickle
import json

trusted = {"numbers": [1, 2, 3]}
pickle_bytes = pickle.dumps(trusted)
json_text = json.dumps(trusted)
print(len(pickle_bytes), json.loads(json_text))
# Never run pickle.loads() on untrusted serialized bytes.

**Design review:** Why isn't pickle appropriate for receiving arbitrary client-uploaded objects?

### 3. Secrets and comparisons

Use secrets for security-sensitive random values. hmac.compare_digest reduces some timing leakage for matching secret values, but it doesn't replace correct password hashing, access controls, or key rotation.

In [ ]:
import secrets
import hmac

token = secrets.token_hex(16)
candidate = token
print("Token length:", len(token))
print("Equal:", hmac.compare_digest(token, candidate))
assert hmac.compare_digest(token, candidate)

**Design review:** Why should access tokens never be printed in tutorial logs?

### 4. Safe subprocess invocation

subprocess.run with a list of arguments and shell=False avoids shell parsing of input. Validate the executable and arguments anyway. This example only constructs an argument list; it executes no external command.

In [ ]:
from pathlib import Path

def build_safe_command(filename):
    path = Path(filename)
    if path.suffix != ".txt":
        raise ValueError("only .txt accepted")
    return ["python", "--version"], path.name

print(build_safe_command("notes.txt"))
# Avoid: subprocess.run(f"somecommand {user_input}", shell=True)

**Design review:** Why can shell=False still be unsafe if you accept any executable name?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/03_security_testing_performance_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/13', {1: 'ecd071350e14d84c', 2: 'a23cf573aebd8913', 3: 'e4aea051b3a5fb59', 4: '2ee1d9ab5f754b73'})

**Exercise 1.** Parse untrusted data with `json`, never `eval`.

In [ ]:
import json
untrusted = '{"user": "ana", "admin": false}'
data = json.___(untrusted)
check(1, data)

**Exercise 2.** Run commands with an argument list, so user input can't inject shell syntax.

In [ ]:
import shlex
user_input = "notes.txt; rm -rf /"
args = ["wc", "-l", ___]
check(2, (len(args), args[-1]))

**Exercise 3.** Generate a token with `secrets` and compare secrets in constant time.

In [ ]:
import hmac, secrets
token = secrets.___(32)
check(3, (len(token) >= 43, hmac.___(token, token), hmac.compare_digest(token, token + "x")))

**Exercise 4.** Bound untrusted input before parsing it.

In [ ]:
import json

def parse_limited(text, max_bytes=100):
    if len(text.encode("utf-8")) > ___:
        raise ValueError("too large")
    return json.loads(text)

try:
    parse_limited('["' + "x" * 200 + '"]')
    result = "parsed"
except ValueError as error:
    result = str(error)
check(4, (result, parse_limited("[1, 2]")))

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** To parse untrusted data, use…

   - **a)** pickle.loads
   - **b)** json.loads with validation
   - **c)** eval

**2.** Why is unpickling untrusted data dangerous?

   - **a)** It's slow
   - **b)** It can execute arbitrary code
   - **c)** It loses precision

**3.** For security tokens, use…

   - **a)** random.random()
   - **b)** time.time()
   - **c)** secrets.token_urlsafe()

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/13', {1: '44b8dfa7dc5c', 2: '270787e9b442', 3: 'dd83297c1e84'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [14 — Advanced Testing: Properties, Integration, Fakes, and CI](14_testing_property_integration.ipynb)